# Lábia — avaliação: linha de base (modelo sem ajuste, prompt padrão e reforçado)

Configurações avaliadas aqui: `base`, `base+reforco`.

Cada configuração atende os **212 cenários de teste** (134 com táticas vistas no treino e 78 com a tática nunca vista), **3 rodadas** cada, com o cliente simulado pelo **Ministral-3-14B**, de outra família de modelos. O veredito sai do verificador determinístico; as métricas com intervalo de confiança são calculadas depois, pelo `labia.evaluate`, a partir dos arquivos que este notebook grava.

Como o piloto mostrou, dois modelos grandes no mesmo servidor ficam se revezando na GPU. Aqui há **dois servidores, um por GPU**: o agente avaliado na GPU 0 e o simulador na GPU 1.

O modelo-base e o modelo ajustado passam pelo **mesmo caminho de conversão** (pesos do Hugging Face → GGUF de 8 bits → Ollama), para a comparação medir o treino, e não diferenças de compressão.

In [ ]:
import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path
LABIA_COMMIT = "9cf4deaddbf7c6440df4b32426242c3408f05ac4"
CONFIGS = [{"model_id": "base", "agent_tag": "labia-base", "source": "base", "defense": "standard"}, {"model_id": "base+reforco", "agent_tag": "labia-base", "source": "base", "defense": "reinforced"}]
SIMULATOR = "ministral-3:14b-instruct-2512-q4_K_M"
BASE_HF = "Qwen/Qwen3-4B-Instruct-2507"
RUNS = 3
WORKERS = 8
MAX_CALLS = 40000
OUT = Path("/kaggle/working/avaliacao"); OUT.mkdir(parents=True, exist_ok=True)
START = time.perf_counter()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-400:])

## 1. Dois servidores, um por GPU

- Agente (porta 11434, GPU 0): janela de 12.288 tokens, porque leva a política, as ferramentas e a conversa inteira.
- Simulador (porta 11435, GPU 1): janela de 6.144 tokens, que sobra para o prompt do cliente e o diálogo, e deixa a memória da conversa de 8 clientes simultâneos caber na T4.

In [ ]:
subprocess.run("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-200:])


def start_server(gpu, port, context):
    models = f"/tmp/ollama-{port}"
    os.makedirs(models, exist_ok=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OLLAMA_HOST=f"127.0.0.1:{port}", OLLAMA_MODELS=models,
               OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_CONTEXT_LENGTH=str(context), OLLAMA_FLASH_ATTENTION="1",
               OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="-1", OLLAMA_MAX_LOADED_MODELS="1")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"/kaggle/working/ollama-{port}.log", "w"),
                     stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/api/tags", timeout=2)
            return env
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"servidor {port} não subiu")


AGENT_ENV = start_server(0, 11434, 12288)
SIM_ENV = start_server(1, 11435, 6144)
subprocess.run(["ollama", "pull", SIMULATOR], env=SIM_ENV, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("servidores no ar; simulador baixado")

## 2. Os agentes avaliados, todos em GGUF de 8 bits

- **base**: os pesos originais do Qwen3-4B-Instruct-2507, convertidos aqui pelo script oficial do llama.cpp.
- **ajustados**: o GGUF que o notebook de treino gravou, já com o adaptador LoRA fundido (anexado como fonte deste notebook).

O Ollama lê o template de chat embutido no GGUF, o mesmo que o treino usou.

In [ ]:
def ensure_llama_cpp():
    if not Path("/tmp/llama.cpp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/tmp/llama.cpp"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/tmp/llama.cpp/gguf-py", "sentencepiece"], check=True)


def base_gguf():
    target = Path("/tmp/base-q8_0.gguf")
    if target.exists():
        return target
    from huggingface_hub import snapshot_download
    ensure_llama_cpp()
    src = snapshot_download(BASE_HF, local_dir="/tmp/base-hf")
    subprocess.run([sys.executable, "/tmp/llama.cpp/convert_hf_to_gguf.py", src, "--outtype", "q8_0", "--outfile", str(target)],
                   check=True, stdout=subprocess.DEVNULL)
    return target


def find_gguf(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} não encontrado em /kaggle/input: anexe o notebook de treino como fonte")
    return hits[0]


def create_model(tag, gguf):
    modelfile = Path(f"/tmp/{tag}.Modelfile")
    modelfile.write_text(f"FROM {gguf}\nPARAMETER num_ctx 12288\n", encoding="utf-8")
    subprocess.run(["ollama", "create", tag, "-f", str(modelfile)], env=AGENT_ENV, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    show = subprocess.run(["ollama", "show", tag], env=AGENT_ENV, capture_output=True, text=True).stdout
    assert "tools" in show, f"{tag}: o Ollama não reconheceu suporte a ferramentas no template\n{show}"
    print(f"{tag}: criado a partir de {gguf.name}")

## 3. Rodar cada configuração

Cada conversa é gravada assim que termina (`{model_id}.jsonl`). Se a sessão cair, rodar de novo retoma de onde parou.

In [ ]:
from collections import Counter
from labia.agents import LLMAgent, LLMCustomer
from labia.contracts.common import Defense
from labia.llm import CachedClient, OpenAICompatClient
from labia.runner import Job, load_records, run_batch, run_seed
from labia.scenarios.generator import generate

test = generate("test")
print(len(test), "cenários de teste;", dict(Counter(s.tactic.value for s in test)))
summary = {}
created = set()
for cfg in CONFIGS:
    tag = cfg["agent_tag"]
    if tag not in created:
        gguf = base_gguf() if cfg["source"] == "base" else find_gguf(cfg["source"])
        create_model(tag, gguf)
        created.add(tag)
    agent_client = CachedClient(OpenAICompatClient("http://127.0.0.1:11434/v1", "ollama"), OUT / "cache" / cfg["model_id"] / "agent", max_calls=MAX_CALLS)
    sim_client = CachedClient(OpenAICompatClient("http://127.0.0.1:11435/v1", "ollama"), OUT / "cache" / cfg["model_id"] / "sim", max_calls=MAX_CALLS)
    start = time.perf_counter()
    run_batch(
        [Job(s, r) for s in test for r in range(1, RUNS + 1)],
        lambda s, r: LLMAgent(agent_client, tag, temperature=0.7, seed=run_seed(s, r, "agent")),
        lambda s, r: LLMCustomer(sim_client, SIMULATOR, s, seed=run_seed(s, r, "customer"), max_tokens=250),
        OUT / f"{cfg['model_id']}.jsonl", model_id=cfg["model_id"], defense=Defense(cfg["defense"]), workers=WORKERS, log_every=60,
    )
    records = load_records(OUT / f"{cfg['model_id']}.jsonl")
    hours = (time.perf_counter() - start) / 3600
    summary[cfg["model_id"]] = {
        "conversas": len(records), "horas": round(hours, 2),
        "vereditos": dict(Counter(r.verdict.value for r in records)),
        "chamadas_validas": round(sum(r.n_valid_calls for r in records) / max(1, sum(r.n_calls for r in records)), 4),
        "fim": dict(Counter(r.ended_by for r in records)),
        "chamadas_ao_modelo": agent_client.calls_made + sim_client.calls_made,
    }
    print(cfg["model_id"], json.dumps(summary[cfg["model_id"]], ensure_ascii=False))
    shutil.rmtree(OUT / "cache" / cfg["model_id"], ignore_errors=True)

summary["_sessao_horas"] = round((time.perf_counter() - START) / 3600, 2)
(OUT / "resumo.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))